# LC 325 — Maximum Size Subarray Sum Equals K
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Prefix Sum
**Pattern:** Prefix Sum + HashMap (Earliest Index)

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Store the FIRST time each
prefix sum appears. When prefix[j] - k is already in
the map, you found a subarray of sum k — and using the
earliest index maximises its length.
</div>

## Official Problem Statement

Given an integer array `nums` and an integer `k`,
return the maximum length of a subarray that sums
to `k`. If there is not one, return `0`.

**Example 1:**
```
Input:  nums = [1,-1,5,-2,3], k = 3
Output: 4
Explanation: subarray [1,-1,5,-2] sums to 3
```
**Example 2:**
```
Input:  nums = [-2,-1,2,1], k = 1
Output: 2
Explanation: subarray [-1,2] sums to 1
```

**Constraints:**
- `1 <= nums.length <= 2 * 10^5`
- `-10^4 <= nums[i] <= 10^4`
- `-10^9 <= k <= 10^9`

## What This Is Actually Asking

Find the longest continuous slice of the array
that adds up to exactly k.
The array can have negatives, so sliding window
won't work here.
We need a smarter way to find distant start points
that still give us sum k.

## Walk Through an Example by Hand

```
nums = [1, -1, 5, -2, 3]    k = 3

Key math: sum(i..j) = prefix[j] - prefix[i-1]
We want sum = k, so: prefix[i-1] = prefix[j] - k

prefix_map = {0: -1}   (sum 0 seen before index 0)
total = 0

i=0  num=1   total=1   need (1-3)=-2  not in map
             store 1->0   map={0:-1, 1:0}

i=1  num=-1  total=0   need (0-3)=-3  not in map
             0 already in map (keep earliest) skip

i=2  num=5   total=5   need (5-3)=2   not in map
             store 5->2   map={0:-1,1:0,5:2}

i=3  num=-2  total=3   need (3-3)=0   FOUND at -1!
             length = 3-(-1) = 4   max_len=4
             store 3->3

i=4  num=3   total=6   need (6-3)=3   FOUND at 3!
             length = 4-3 = 1   max_len stays 4

Answer: 4
```

## The Picture

```
Array:    [  1,  -1,   5,  -2,   3 ]
index:       0    1    2    3    4

Prefix:   0    1    0    5    3    6
         -1    0    1    2    3    4   <- indices
          ^
        seed: prefix 0 at index -1

At index 3: prefix = 3.  3 - k = 3 - 3 = 0
  0 is in the map at index -1
  subarray from index 0..3 -> length = 3-(-1) = 4

  [=====1=====|-1|=====5=====|-2=====]
   start                        end
   <- length 4, sums to 3 ->

Rule: store FIRST occurrence so the span is longest.
```

## When To Use This Pattern

- When you see **subarray sum = k with negatives**,
  think **prefix sum + HashMap** (not sliding window)
- When you see **longest subarray**, think
  **store only the FIRST index for each prefix sum**
- When you see **sum(i..j) = k**, think
  **prefix[j] - prefix[i-1] = k** and rearrange
- When seeding the map, think
  **always start with {0: -1}** to handle subarrays
  that begin at index 0

## The Approach

Seed a HashMap with `{0: -1}` — prefix sum zero
exists before the array starts.
Walk the array keeping a running total.
At each index, check if `total - k` is in the map —
if yes, a subarray of sum k ends here; update max
length using the stored (earliest) index.
Store the current total in the map only if it has
not been seen before, to keep the earliest index.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    tests = [
        # (nums, k, expected_length)
        ([1,-1,5,-2,3],   3,  4),
        ([-2,-1,2,1],     1,  2),
        ([1,2,3],         3,  2),   # [1,2] and [3] both work; [1,2] is longer? No, same length
        ([1,2,3],         6,  3),   # whole array
        ([1,2,3],         7,  0),   # not found
        ([0,0,0],         0,  3),   # zeros, longest is full array
        ([-1,-1,1,1,1],   0,  4),   # negatives cancel
        ([1],             1,  1),   # single element match
        ([1],             2,  0),   # single element no match
        ([1,-1,1,-1,1],   1,  5),   # alternating, full array
    ]

    passed = 0
    for i, (nums, k, expected) in enumerate(tests):
        result = func(nums[:], k)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | k={k} | "
            f"nums={nums} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def maxSubArrayLen(nums: List[int], k: int) -> int:
    """
    Return the length of the longest subarray summing
    to k.

    Seed map with {0: -1}. Walk array tracking running
    total. At each index, if total-k is in the map,
    compute the span. Store each prefix sum only on its
    FIRST appearance so the stored index is as early
    as possible, maximising the span.

    Time:  O(n) — single pass with O(1) hash lookups
    Space: O(n) — map holds at most n+1 prefix sums
    """
    pass


# Quick debug — run this cell while building
print(maxSubArrayLen([1,-1,5,-2,3], 3))   # 4
print(maxSubArrayLen([-2,-1,2,1], 1))     # 2
print(maxSubArrayLen([1,2,3], 7))         # 0
print(maxSubArrayLen([0,0,0], 0))         # 3

In [ ]:
# Uncomment and run when solution is ready
# test_harness(maxSubArrayLen)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — check all subarrays | O(n²) | O(1) |
| Prefix sum + HashMap | O(n) | O(n) |

The HashMap turns every "find a past prefix sum"
lookup from O(n) into O(1) — dropping the whole
solution from quadratic to linear.

## Real World Connection

At Citi, daily P&L for a trading desk can be positive
or negative — exactly like this array with negatives.
Finding the longest consecutive run of trading days
where net P&L hits a target `k` tells the risk team
which sustained periods contributed that exact amount.
A brute-force scan of all day windows is O(n²) and
too slow across years of data; the prefix sum map
finds the answer in one O(n) pass.
The same pattern appears in ETL pipeline monitoring:
finding the longest window of time-delta metrics
that sum to a target processing latency threshold.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra